# Exploração do dataset PlantVillage

Este notebook será utilizado na etapa de **compreensão e preparação dos dados** do TCC.

Nesta primeira parte, vamos apenas:

- montar o Google Drive;
- criar a pasta do projeto;
- baixar os arquivos oficiais do PlantVillage;
- baixar o mapa de agrupamento das folhas;
- verificar se os arquivos foram obtidos corretamente.

**Ainda não será feita divisão treino/validação/teste nem treinamento de modelos.**


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TCC")
DATA_DIR = BASE_DIR / "data"

BASE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Pasta base:", BASE_DIR)
print("Pasta dos dados:", DATA_DIR)


Pasta base: /content/drive/MyDrive/TCC
Pasta dos dados: /content/drive/MyDrive/TCC/data


## Download dos arquivos oficiais

O repositório do PlantVillage disponibiliza um arquivo `data.zip` com as imagens e um arquivo
`leaf_grouping/leaf-map.json` com informações usadas para agrupar imagens derivadas da mesma folha.

Esse agrupamento será importante posteriormente para criar a divisão **70% / 15% / 15%** sem
vazamento de dados entre treinamento, validação e teste.


In [3]:
!pip install -q huggingface_hub


In [4]:
from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="data.zip",
    repo_type="dataset",
    local_dir=str(DATA_DIR)
)

leaf_map_path = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="leaf_grouping/leaf-map.json",
    repo_type="dataset",
    local_dir=str(DATA_DIR)
)

print("Arquivo de imagens:", zip_path)
print("Mapa de folhas:", leaf_map_path)


data.zip:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

leaf-map.json:   0%|          | 0.00/2.43M [00:00<?, ?B/s]

Arquivo de imagens: /content/drive/MyDrive/TCC/data/data.zip
Mapa de folhas: /content/drive/MyDrive/TCC/data/leaf_grouping/leaf-map.json


## Verificação do download


In [5]:
import os

zip_size_gb = os.path.getsize(zip_path) / (1024 ** 3)

print(f"Tamanho do data.zip: {zip_size_gb:.2f} GB")
print("data.zip existe:", os.path.exists(zip_path))
print("leaf-map.json existe:", os.path.exists(leaf_map_path))


Tamanho do data.zip: 2.03 GB
data.zip existe: True
leaf-map.json existe: True


In [6]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    arquivos = z.namelist()

print("Quantidade total de arquivos no ZIP:", len(arquivos))

print("\nPrimeiros 30 arquivos:")
for arquivo in arquivos[:30]:
    print(arquivo)

Quantidade total de arquivos no ZIP: 163034

Primeiros 30 arquivos:
raw/
raw/color/
raw/color/Raspberry___healthy/
raw/color/Raspberry___healthy/6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL 6318.JPG
raw/color/Raspberry___healthy/d1387960-14d6-4654-ad5c-b92afa86ff6a___Mary_HL 9262.JPG
raw/color/Raspberry___healthy/fafac0a9-e5ba-420d-9dbe-034942f845b9___Mary_HL 9201.JPG
raw/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG
raw/color/Raspberry___healthy/fe1c1683-06cc-49c5-b86a-a41fb36f058b___Mary_HL 6249.JPG
raw/color/Raspberry___healthy/e621e3c3-842a-4489-b931-89f05c254cbe___Mary_HL 9347.JPG
raw/color/Raspberry___healthy/b8a96dea-8442-4855-9fdf-c29c9d4eca5a___Mary_HL 6398.JPG
raw/color/Raspberry___healthy/42428dcc-fefc-46dd-b209-150fbeaf4862___Mary_HL 6397.JPG
raw/color/Raspberry___healthy/076d909d-cfca-47ce-a8d1-479bb77c4bab___Mary_HL 6294.JPG
raw/color/Raspberry___healthy/27ef7a65-771c-4eab-9169-03a5fa121c71___Mary_HL 9195.JPG
raw/color/Raspberry___hea

In [7]:
arquivos_color = [
    arquivo for arquivo in arquivos
    if "raw/color" in arquivo.lower()
]

print("Arquivos em raw/color:", len(arquivos_color))

for arquivo in arquivos_color[:10]:
    print(arquivo)

Arquivos em raw/color: 54344
raw/color/
raw/color/Raspberry___healthy/
raw/color/Raspberry___healthy/6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL 6318.JPG
raw/color/Raspberry___healthy/d1387960-14d6-4654-ad5c-b92afa86ff6a___Mary_HL 9262.JPG
raw/color/Raspberry___healthy/fafac0a9-e5ba-420d-9dbe-034942f845b9___Mary_HL 9201.JPG
raw/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG
raw/color/Raspberry___healthy/fe1c1683-06cc-49c5-b86a-a41fb36f058b___Mary_HL 6249.JPG
raw/color/Raspberry___healthy/e621e3c3-842a-4489-b931-89f05c254cbe___Mary_HL 9347.JPG
raw/color/Raspberry___healthy/b8a96dea-8442-4855-9fdf-c29c9d4eca5a___Mary_HL 6398.JPG
raw/color/Raspberry___healthy/42428dcc-fefc-46dd-b209-150fbeaf4862___Mary_HL 6397.JPG


In [8]:
import json

with open(leaf_map_path, "r", encoding="utf-8") as f:
    leaf_map = json.load(f)

print("Tipo:", type(leaf_map))
print("Quantidade:", len(leaf_map))

if isinstance(leaf_map, dict):
    print("\nPrimeiros itens:")
    for i, (chave, valor) in enumerate(leaf_map.items()):
        print(chave, "->", valor)
        if i == 9:
            break

elif isinstance(leaf_map, list):
    print("\nPrimeiros itens:")
    for item in leaf_map[:10]:
        print(item)

Tipo: <class 'dict'>
Quantidade: 40328

Primeiros itens:
rutg._hl 3733 -> ['Peach___healthy:::54.0']
com.g_fl 8310 -> ['Tomato___Target_Spot:::243.0']
rs_early.b 7557 -> ['Potato___Early_blight:::115.0']
rs_early.b 7554 -> ['Potato___Early_blight:::115.0']
uf.citrus_hlb_lab 1658 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
uf.citrus_hlb_lab 1659 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
uf.citrus_hlb_lab 1654 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
uf.citrus_hlb_lab 1655 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
uf.citrus_hlb_lab 1656 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
uf.citrus_hlb_lab 1657 -> ['Orange___Haunglongbing_(Citrus_greening):::104.0']
